In [1]:
from rxn4chemistry import RXN4ChemistryWrapper

# 替换为你的实际 API Key
API_KEY = "apk-939cbd36c7866253b729a8d077061e9f86e3a60ac1a6cb4d0042ba8dcdcffa99"

# 初始化客户端
rxn = RXN4ChemistryWrapper(api_key=API_KEY)

# 创建新项目（如果已有项目 ID 可跳过）
response = rxn.create_project(name="Hesperidin_Retrosynthesis")
print(response)

# 记录项目 ID
PROJECT_ID = response['response']['payload']['id']
print(f"项目 ID: {PROJECT_ID}")

{'response': {'metadata': {'extendedPagination': {}, 'uiMessages': {'errors': [], 'infos': [], 'warnings': []}}, 'payload': {'attempts': [], 'computedFields': {}, 'createdBy': '841e526f-4c88-4f30-8a65-9be7ff586916', 'createdOn': '2026-06-04T11:09:57.994Z', 'description': None, 'embed': {}, 'id': '6a215d057f6b9b3ec2bc32ed', 'isFavorite': False, 'metadata': {}, 'modifiedBy': '841e526f-4c88-4f30-8a65-9be7ff586916', 'modifiedOn': '2026-06-04T11:09:57.994Z', 'name': 'Hesperidin_Retrosynthesis', 'visibility': None}}}
项目 ID: 6a215d057f6b9b3ec2bc32ed


In [6]:
# 列出所有项目
projects = rxn.list_all_projects()
for project in projects['payload']['content']:
    print(f"项目名: {project['name']}, ID: {project['id']}")

KeyError: 'payload'

In [5]:
from rdkit import Chem
from rdkit.Chem import AllChem
hesperidin = Chem.MolFromMolFile('Hesperidin.sdf')
Chem.MolToSmiles(hesperidin)

'COc1ccc([C@@H]2CC(=O)c3c(O)cc(O[C@@H]4O[C@H](CO[C@@H]5O[C@@H](C)[C@H](O)[C@@H](O)[C@H]5O)[C@@H](O)[C@H](O)[C@H]4O)cc3O2)cc1O'

In [8]:
from rxn4chemistry import RXN4ChemistryWrapper
import time

API_KEY = "apk-939cbd36c7866253b729a8d077061e9f86e3a60ac1a6cb4d0042ba8dcdcffa99"
PROJECT_ID = "6a215d057f6b9b3ec2bc32ed"

rxn = RXN4ChemistryWrapper(api_key=API_KEY)
rxn.set_project(PROJECT_ID)

hesperidin_smiles = Chem.MolToSmiles(hesperidin)

# 提交逆合成预测请求
print("正在提交逆合成预测请求...")
response = rxn.predict_automatic_retrosynthesis(
    product=hesperidin_smiles,
    max_steps=5,           # 最大步数
    nbeams=10,             # 搜索束宽
    pruning_steps=2,       # 剪枝步数
    ai_model="2020-07-01"  # 模型版本
)

# 获取预测任务 ID
prediction_id = response['response']['payload']['id']
print(f"预测任务 ID: {prediction_id}")

# 保存任务 ID 供后续查询
with open("prediction_id.txt", "w") as f:
    f.write(prediction_id)

# 保存任务 ID 供后续查询
with open("prediction_id.txt", "w") as f:
    f.write(prediction_id)

正在提交逆合成预测请求...
预测任务 ID: 6a215ee87f6b9b3ec2bc37b7


In [9]:
import time

print("等待逆合成计算完成...")
print("通常需要 30 秒到 5 分钟，取决于分子复杂度")

max_wait = 600  # 最大等待 10 分钟
waited = 0
interval = 10   # 每 10 秒查询一次

while waited < max_wait:
    result = rxn.get_predict_automatic_retrosynthesis_results(prediction_id)
    status = result['response']['payload']['status']
    
    print(f"[{waited}s] 状态: {status}")
    
    if status == 'SUCCESS':
        print("计算完成！")
        break
    elif status in ['FAILED', 'ERROR']:
        print(f"计算失败: {result}")
        break
    
    time.sleep(interval)
    waited += interval
else:
    print("等待超时，请稍后手动查询结果")

# 保存完整结果
import json
with open("retrosynthesis_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print("结果已保存到 retrosynthesis_result.json")

等待逆合成计算完成...
通常需要 30 秒到 5 分钟，取决于分子复杂度
[0s] 状态: PROCESSING
[10s] 状态: PROCESSING
[20s] 状态: PROCESSING
[30s] 状态: PROCESSING
[40s] 状态: PROCESSING
[50s] 状态: PROCESSING
[60s] 状态: PROCESSING
[70s] 状态: PROCESSING
[80s] 状态: PROCESSING
[90s] 状态: PROCESSING
[100s] 状态: SUCCESS
计算完成！
结果已保存到 retrosynthesis_result.json


In [ ]:
import json

with open("retro_synthesis/retrosynthesis_result.jsonretrosynthesis_result.json", "r", encoding="utf-8") as f:
    result = json.load(f)

payload = result['response']['payload']

# 检查是否有结果
if 'retrosynthetic_paths' not in payload or not payload['retrosynthetic_paths']:
    print("未找到逆合成路径，可能原因：")
    print("- 分子过于复杂，超出模型能力范围")
    print("- 模型未覆盖相关反应类型（如糖基化）")
    print("建议尝试更简单的分子或调整参数")
else:
    paths = payload['retrosynthetic_paths']
    print(f"共找到 {len(paths)} 条逆合成路径\n")
    
    for i, path in enumerate(paths[:5], 1):  # 只展示前 5 条
        print(f"=== 路径 {i} ===")
        print(f"置信度: {path.get('confidence', 'N/A')}")
        print(f"步数: {len(path.get('reactions', []))}")
        
        # 遍历每一步反应
        for step_num, reaction in enumerate(path.get('reactions', []), 1):
            print(f"\n  步骤 {step_num}:")
            print(f"    反应 SMILES: {reaction.get('smiles', 'N/A')}")
            print(f"    反应名称: {reaction.get('name', 'N/A')}")
            print(f"    置信度: {reaction.get('confidence', 'N/A')}")
            
            # 反应物
            reactants = reaction.get('reactants', [])
            print(f"    反应物: {', '.join(reactants)}")
            
            # 产物
            products = reaction.get('products', [])
            print(f"    产物: {', '.join(products)}")
        
        print("\n" + "-" * 50)

未找到逆合成路径，可能原因：
- 分子过于复杂，超出模型能力范围
- 模型未覆盖相关反应类型（如糖基化）
建议尝试更简单的分子或调整参数


In [11]:
from rdkit import Chem
from rdkit.Chem import Draw

def visualize_reaction(reaction_smiles, filename="reaction.png"):
    """将反应 SMILES 转换为可视化图片"""
    try:
        rxn = Chem.rdChemReactions.ReactionFromSmarts(reaction_smiles, useSmiles=True)
        if rxn is None:
            print(f"无法解析反应: {reaction_smiles}")
            return
        
        img = Draw.ReactionToImage(rxn, subImgSize=(300, 300))
        img.save(filename)
        print(f"反应图已保存: {filename}")
    except Exception as e:
        print(f"可视化失败: {e}")

# 示例：可视化第一条路径的第一步
if paths:
    first_path = paths[0]
    if first_path.get('reactions'):
        first_reaction = first_path['reactions'][0]
        rxn_smiles = first_reaction.get('smiles', '')
        if rxn_smiles:
            visualize_reaction(rxn_smiles, "step1_reaction.png")

NameError: name 'paths' is not defined